In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime , timedelta
import pandas as pd
import matplotlib.dates as mdates
import io
from PIL import Image
from folium.plugins import HeatMap
import folium
import os
import colorgram 
import plotly.express as px
from PIL import Image
from io import BytesIO
import requests


figures_path = os.path.join('figures')

monthly_metrics_path = 'data/monthly_metrics.csv'
observations_path = 'data/observations.csv'
new_species_path = 'data/new_species_photo.csv'
taxon_counts_path = 'data/taxon_counts.csv'
# 

palette_orange = ['#f85532'] 
['#2b2e4f', '#f85532', '#fe5631', '#f95532', '#2a2e4f']

['#2b2e4f', '#f85532', '#fe5631', '#f95532', '#2a2e4f']

# 1.Graficación y muestra de métricas principales

1.1. número de observaciones, número de observadores, identificadores y especies
1.2. variación respecto al mes anterior

Estas métricas se mostrarán en forma de tabla debido a que es la mejor opción disponible 

# 2.Graficación evolución mensual de las metricas principales

Gráficos por observaciones, observadores, idntificadores y especies

In [ ]:
df_monthly = pd.read_csv('data\monthly_metrics.csv')

In [ ]:
def plot_monthly_metrics(df_monthly, column:str):

    fig = px.bar(df_monthly, 
                x='month', 
                y=column,
                text=column,  # Esto añade los valores como texto
                title=f'Evolució mensual de {column}',
                #labels={'month': 'Mes', 'observations': 'Número de Observaciones'},
                )  

    fig.update_traces(
        #texttemplate='%{text}',  # Formato del texto (puedes usar '%{text:.0f}' para números enteros)
        marker_color='#f85532',
        textposition='outside'
        )  # Posición fuera de las barras

    # Personalizar el diseño si lo deseas
    fig.update_layout(
        title_x=0.5,  # Centrar el título
        xaxis_title="Mes",
        yaxis_title= column,
        template="plotly_white", # Puedes cambiar el template (plotly, plotly_white, plotly_dark, etc.)
        yaxis=dict(
            range=[0, max(df_monthly[column]) * 1.15],  # Espacio extra para el texto
            gridcolor='lightgray'  # Color de la cuadrícula
        ),
        
        
    )

    fig.show()
'''
    os.makedirs("figures", exist_ok=True)
    
    # Guardar como HTML
    html_path = f"figures/{column}_monthly_plot.html"
    fig.write_html(html_path)

    fig.show()

    # Guardar como imagen PNG
    img_bytes = fig.to_image(format="png", scale=3)
    img = Image.open(io.BytesIO(img_bytes))
    img.save(f'figures/monthly_plot.png')

'''


In [ ]:
plot_monthly_metrics(df_monthly, 'species')

In [ ]:
plot_monthly_metrics(df_monthly, 'observations')

In [ ]:
plot_monthly_metrics(df_monthly, 'observers')

In [ ]:
plot_monthly_metrics(df_monthly, 'identifiers')

# 3. Graficación taxonomias

Graficación nuevas especies 

In [ ]:
df_new_species = pd.read_csv('data/new_species_photos.csv')

In [ ]:
def plot_new_species(df_new_species ):

        image_col = 'photos_medium_url'     
        name_col = 'taxon_name'          
        #date_col = 'observed_on'           
        user_col = 'user_login'
        attribution_col = 'attribution'         

        # Mostrar las imágenes
        for index, row in df_new_species.iterrows():
                
                response = requests.get(row[image_col])
                img = Image.open(BytesIO(response.content))
                
                fig = plt.figure(figsize=(5, 5))
                #ax = fig.add_axes([0, 0.05, 1, 0.9])  # [left, bottom, width, height]
                #ax.imshow(img)
                #ax.axis('off')
                plt.imshow(img)
                plt.axis('off')
                plt.suptitle(f"{row[name_col]} \n Usuari: {row[user_col]}", fontsize = 12, ha='center')
                plt.figtext(0.5,0.01, f'{row[attribution_col]}', ha='right', fontsize=8, style = 'italic')               
                plt.tight_layout()
                plt.show()

In [ ]:
plot_new_species( df_new_species )

Top 10 especies 

In [2]:
df_taxon_count = pd.read_csv(taxon_counts_path)

In [ ]:
def plot_top_species(df_taxon_count, taxon_rank:str):
    
    top_df = df_taxon_count[df_taxon_count['taxon_rank'] == taxon_rank]\
             .sort_values('count', ascending=False)\
             .head(10)
    

    fig = px.bar(top_df,
             x='count',
             y=taxon_rank,
             orientation='h',  # Barras horizontales
             text='count',     # Muestra los valores en las barras
             title='Top 10 especies más observadas',
             labels={'count': 'Número de observaciones', 'taxon_name': 'Especie'},
             color_discrete_sequence=['#f85532'])

    fig.update_traces(texttemplate='%{text:,}',  # Formato con separadores de miles
                  textposition='outside',)   # Texto fuera de las barras
                  #marker_line_color='black', # Borde negro en las barras
                  #marker_line_width=1

    fig.update_layout(
    yaxis={'categoryorder':'total ascending'},  # Ordena de mayor a menor
    plot_bgcolor='white',                      
    xaxis_range=[0, top_df['count'].max() * 1.1],  # Margen para el texto
    showlegend=False,
    coloraxis_showscale=False  # Oculta la barra de color si no la necesitas
)

# Guardar como imagen
    #fig.write_image('figures/top_10_especies.png', scale=2, width=1000, height=600)

# Mostrar el gráfico
    fig.show()

In [12]:
plot_top_species(df_taxon_count,'kingdom')

ValueError: Value of 'y' is not the name of a column in 'data_frame'. Expected one of ['taxon_rank', 'taxon_name', 'count'] but received: kingdom

# 5. Mapa calor para densidad de observaciones

In [ ]:
def get_heatmap(zoom_start:int):

    csv_obs = pd.read_csv(observations_path)

    df_valid = csv_obs.dropna(subset=["latitude", "longitude"])
    heat_data = df_valid[["latitude", "longitude"]].values.tolist()

    mean_lat = df_valid["latitude"].mean()
    mean_lon = df_valid["longitude"].mean()
   

    m = folium.Map(location=[mean_lat, mean_lon], zoom_start=zoom_start)
    HeatMap(heat_data).add_to(m)

    os.makedirs("figures", exist_ok=True)

    html_path = "figures/heatmap.html"
    m.save(html_path)

    img_data = m._to_png(3)
    img = Image.open(io.BytesIO(img_data))
    
    img.save('figures/heatmap_image.png')

In [ ]:
get_heatmap(11)